# 05 — X-IIoTID Model Evaluation

**Goal:** Compare the five trained models and examine where detection errors happen.

**Input:** Saved Notebook 04 models, validation data and model reports.

**Output:** Model comparison, attack-category recall, missing-value checks and two graphs.

**Rule:** Validation only. No training, no test access.

In [ ]:
from pathlib import Path
import gc, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.metrics import confusion_matrix

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
MODELS = ROOT / 'models/x_iiotid_v3'
VALIDATION = ROOT / 'data/splits/x_iiotid/x_iiotid_core19_validation_v3.csv.gz'
SPLIT_REPORT = AUDITS / 'x_iiotid_v3_split_report.json'
MODEL_NAMES = ('decision_tree','random_forest','extra_trees','lightgbm','adaboost')
MISSING = {'','-','?','na','n/a','nan','none','null','missing','unknown','<missing>'}
CAT = ['Protocol','Service','Conn_state','is_syn_only','Is_SYN_ACK',
       'is_pure_ack','is_with_payload','FIN or RST']

for path in (VALIDATION,SPLIT_REPORT):
    if not path.is_file(): raise FileNotFoundError(path)
for folder in (METRICS, FIGURES): folder.mkdir(parents=True, exist_ok=True)
split_info = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
if split_info['counts'] != {'train':574585,'validation':123125,'test':123124}:
    raise ValueError('Unexpected frozen split counts')
if split_info.get('normalized_signature_overlap_groups') != 0:
    raise ValueError('Frozen feature-signature separation not verified')
FEATURES = split_info['features']
NUM = [col for col in FEATURES if col not in CAT]
if len(FEATURES) != 19 or len(NUM) != 11: raise ValueError('Core19 feature mismatch')
print('Model evaluation: 123,125 validation rows | 19 features | TEST NOT READ')

## 1. Compare saved model results

Recalculate scores from the saved confusion matrices. Models are not retrained.

In [ ]:
rows = []
resources = []
for name in MODEL_NAMES:
    path = AUDITS / f'x_iiotid_v3_{name}_training.json'
    if not path.is_file(): raise FileNotFoundError(f'Run Notebook 04 model section: {name}')
    r = json.loads(path.read_text(encoding='utf-8'))
    m = r['metrics']; cost = r['resources']
    if r.get('test_scored') is not False:
        raise ValueError(f'{name}: unexpected test evaluation flag')
    if (int(m['train_rows']),int(m['validation_rows'])) != (574585,123125):
        raise ValueError(f'{name}: wrong partition sizes')
    tn,fp,fn,tp = [int(m[key]) for key in ('tn','fp','fn','tp')]
    assert tn+fp+fn+tp == 123125
    f1 = (2*tp)/(2*tp+fp+fn)
    fpr = fp/(tn+fp)
    if abs(f1-float(m['f1_attack'])) > 1e-6 or abs(fpr-float(m['false_positive_rate'])) > 1e-6:
        raise ValueError(f'{name}: saved metrics do not match confusion counts')
    rows.append({'model':name,'f1_pct':round(100*f1,2),
                 'false_positive_pct':round(100*fpr,2),'missed_attacks':fn,
                 'false_alarms':fp,'accuracy_pct':round(100*(tn+tp)/123125,2)})
    resources.append({'model':name,
        'p95_ms':float(cost['single_flow_p95_ms']),
        'model_mib':float(cost['model_size_mib']),
        'training_sec':float(cost['training_seconds']),
        'training_peak_process_ram_mib':float(cost['train_peak_process_ram_mib']),
        'training_cpu_pct':float(cost['train_process_cpu_percent'])})
comparison = pd.DataFrame(rows).merge(pd.DataFrame(resources),on='model')
display(comparison)
print('Resource measurements were taken on a laptop, not a Raspberry Pi.')

## 2. Review errors by attack category

Check which attack categories each model misses. Categories come from the original validation labels.

In [ ]:
frame = pd.read_csv(VALIDATION,dtype=str,keep_default_na=False,low_memory=False)
fields = ['csv_row_index_0based'] + FEATURES + ['class1','class2','class3']
if frame.columns.tolist() != fields or len(frame) != 123125:
    raise ValueError('Validation dataset has changed or is missing columns')
if not frame['csv_row_index_0based'].is_unique:
    raise ValueError('Duplicate validation row IDs')
if not frame['class3'].isin(['Normal','Attack']).all():
    raise ValueError('Unexpected validation labels')

def model_features(df):
    data = df[FEATURES].copy()
    for column in NUM:
        values = data[column].astype(str).str.strip()
        values = values.mask(values.str.casefold().isin(MISSING))
        data[column] = pd.to_numeric(values,errors='coerce').replace([np.inf,-np.inf],np.nan).astype(np.float32)
    for column in CAT:
        values = data[column].astype(str).str.strip()
        data[column] = values.mask(values.str.casefold().isin(MISSING),'__MISSING__')
    return data

X_valid = model_features(frame)
y_true = (frame['class3'] == 'Attack').astype(np.uint8).to_numpy()
attack_mask = y_true == 1
category_rows=[]
missing_rows=[]
missing_by = {
    name: frame[name].astype(str).str.strip().str.casefold().isin(MISSING).to_numpy()
    for name in ('Des_bytes','Duration')
}
for name in MODEL_NAMES:
    model_path = MODELS / f'{name}_core19_v3.joblib'
    if not model_path.is_file(): raise FileNotFoundError(f'Missing trained model: {model_path}')
    model = joblib.load(model_path)
    pred = np.asarray(model.predict(X_valid),dtype=np.uint8)
    tn,fp,fn,tp = [int(x) for x in confusion_matrix(y_true,pred,labels=[0,1]).ravel()]
    expected = next(r for r in rows if r['model']==name)
    if fp!=expected['false_alarms'] or fn!=expected['missed_attacks']:
        raise ValueError(f'{name}: saved model predictions disagree with Notebook 04 results')
    for cat, idx in frame.loc[attack_mask].groupby('class2',sort=True).indices.items():
        # groupby indices are positions in the attack-only frame; map back to original positions.
        attack_positions = np.flatnonzero(attack_mask)[idx]
        count = len(attack_positions)
        misses = int((pred[attack_positions] == 0).sum())
        category_rows.append({'model':name,'class2':str(cat),'attack_count':count,
                              'detected':count-misses,'missed':misses,
                              'recall_pct':round(100*(count-misses)/count,2)})
    for feature, missing_flag in missing_by.items():
        for status, selector in [('missing',missing_flag),('present',~missing_flag)]:
            mask = attack_mask & selector
            total = int(mask.sum())
            hits = int((pred[mask]==1).sum())
            missing_rows.append({'model':name,'feature':feature,'group':status,
                                 'attack_count':total,
                                 'recall_pct':round(100*hits/total,2) if total else np.nan})
    del model, pred
    gc.collect()
category_result=pd.DataFrame(category_rows)
missing_result=pd.DataFrame(missing_rows)
print('Attack recall by class2 (showing categories with most records, top three models):')
top_categories = (category_result[category_result.model=='lightgbm']
                  .nlargest(10,'attack_count')['class2'])
display(category_result[
    category_result.model.isin(['lightgbm','decision_tree','random_forest']) &
    category_result.class2.isin(top_categories)
].sort_values(['class2','model']))
print('Detection recall for attack records with missing vs present values:')
display(missing_result[missing_result.model.isin(['lightgbm','decision_tree','random_forest'])])

## 3. Save the evaluation evidence

Keep the outputs small. The test set is still untouched.

In [ ]:
comparison.to_csv(METRICS/'x_iiotid_v3_model_evaluation.csv',index=False)
category_result.to_csv(METRICS/'x_iiotid_v3_attack_category_recall.csv',index=False)
missing_result.to_csv(METRICS/'x_iiotid_v3_missingness_recall.csv',index=False)

# F1 vs false alarms: upper-left is preferred (higher F1 and lower FPR).
fig,ax=plt.subplots(figsize=(8,5))
for r in comparison.itertuples(index=False):
    ax.scatter(r.false_positive_pct,r.f1_pct,s=65)
    ax.annotate(r.model.replace('_',' ').title(),
                (r.false_positive_pct,r.f1_pct),xytext=(5,5),textcoords='offset points',fontsize=9)
ax.set_title('X-IIoTID — Detection versus false alarms (validation)')
ax.set_xlabel('False-positive rate (%)')
ax.set_ylabel('Attack F1-score (%)')
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(FIGURES/'x_iiotid_v3_evaluation_tradeoff.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

# Show LightGBM recall across attack categories; very small classes should be read with care.
view=(category_result[category_result.model=='lightgbm']
      .sort_values('attack_count',ascending=False).head(12).sort_values('recall_pct'))
fig,ax=plt.subplots(figsize=(9,max(4,0.4*len(view)+1)))
ax.barh(view['class2'],view['recall_pct'])
ax.set_xlim(0,105)
ax.set_xlabel('Attack recall (%)')
ax.set_title('LightGBM — recall by class2 attack category (validation)')
ax.grid(axis='x',alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES/'x_iiotid_v3_category_recall.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

summary = {
    'dataset':'X-IIoTID','feature_set':'Core19','model_count':5,
    'training_rows':574585,'validation_rows':123125,'test_scored':False,
    'provisional_shortlist':['lightgbm','decision_tree','random_forest'],
    'comparison':comparison.to_dict(orient='records'),
    'limitations':[
        'Validation-only evidence; held-out test set unopened.',
        'Missing-value patterns and Service may create dataset shortcuts.',
        'No proof of capture-session or campaign independence.',
        'CPU, latency and memory are laptop measurements, not live gateway benchmarks.',
        'Per-category class2 recall measures flow labels, not complete APT campaigns.'
    ]
}
(AUDITS/'x_iiotid_v3_model_evaluation.json').write_text(
    json.dumps(summary,indent=2),encoding='utf-8')
print('Saved three comparison CSV files, two graphs, and one JSON report.')
print('No model training. Test data was not opened.')